In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/nppe-2-t-2-26-ecg-heartbeat-arrhythmia-classification/nppe2_dataset/sample_submission.csv
/kaggle/input/competitions/nppe-2-t-2-26-ecg-heartbeat-arrhythmia-classification/nppe2_dataset/train.csv
/kaggle/input/competitions/nppe-2-t-2-26-ecg-heartbeat-arrhythmia-classification/nppe2_dataset/test.csv


In [2]:
import os
import sys
import subprocess
import random
import gc
import math
import time
import shutil
import warnings
from pathlib import Path
warnings.filterwarnings("ignore")
if "torch" not in sys.modules:
    print("=" * 80)
    print("INSTALLING P100-COMPATIBLE PYTORCH")
    print("=" * 80)
    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "--force-reinstall",
        "torch==2.10.0",
        "--index-url",
        "https://download.pytorch.org/whl/cu126",
        "--no-deps",
    ]
    result = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(
            "PyTorch cu126 installation failed."
        )
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import (
    Dataset,
    DataLoader
)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    f1_score,
    classification_report,
    confusion_matrix
)
SEED = 42
N_FOLDS = 3
EPOCHS = 18
PATIENCE = 4
BATCH_SIZE = 192
NUM_WORKERS = min(
    4,
    os.cpu_count() or 2
)

LR = 2e-3
WEIGHT_DECAY = 2e-4
LABEL_SMOOTHING = 0.04
AMP = True
GRAD_CLIP = 1.0
AUGMENT = True
TTA = True
DEBUG_RUN = False
DEBUG_ROWS_PER_CLASS = 1500
DEBUG_EPOCHS = 2
WORK_DIR = Path(
    "/kaggle/working"
)

SUBMISSION_PATH = (
    WORK_DIR
    / "submission.csv"
)
OLD_DIRS = [
    WORK_DIR / "ecg_pytorch_p100_v1",
    WORK_DIR / "ecg_pytorch_p100_v2",
    WORK_DIR / "ecg_pytorch_p100_v3",
    WORK_DIR / "ecg_pytorch_p100_msconv_transformer_v1",
]
for old_dir in OLD_DIRS:

    if old_dir.exists():

        shutil.rmtree(
            old_dir,
            ignore_errors=True
        )
if SUBMISSION_PATH.exists():
    SUBMISSION_PATH.unlink()

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)
if DEVICE.type != "cuda":

    raise RuntimeError(
        "CUDA is unavailable. "
        "Enable a GPU in Kaggle."
    )
GPU_NAME = (
    torch.cuda.get_device_name(0)
)
GPU_CAPABILITY = (
    torch.cuda.get_device_capability(0)
)
print(
    "Device:",
    DEVICE
)
print(
    "GPU:",
    GPU_NAME
)

print(
    "Compute capability:",
    GPU_CAPABILITY
)
torch.backends.cudnn.benchmark = True

if hasattr(
    torch.backends.cuda,
    "matmul"
):
    torch.backends.cuda.matmul.allow_tf32 = True

if hasattr(
    torch,
    "set_float32_matmul_precision"
):
    torch.set_float32_matmul_precision(
        "high"
    )
def seed_everything(
    seed=SEED
):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything()

print(
    "\nRunning CUDA smoke test..."
)


try:

    test_x = torch.randn(
        16,
        8,
        250,
        device=DEVICE
    )

    test_conv = nn.Conv1d(
        8,
        16,
        5,
        padding=2
    ).to(DEVICE)

    test_y = test_conv(
        test_x
    )
    test_loss = (
        test_y.square().mean()
    )
    test_loss.backward()
    torch.cuda.synchronize()

    del (
        test_x,
        test_conv,
        test_y,
        test_loss
    )

    torch.cuda.empty_cache()

    print(
        "CUDA SMOKE TEST: PASSED"
    )

except Exception as e:

    raise RuntimeError(
        "CUDA smoke test failed:\n"
        + repr(e)
    )

INSTALLING P100-COMPATIBLE PYTORCH
Looking in indexes: https://download.pytorch.org/whl/cu126
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 841.9/841.9 MB 339.5 MB/s eta 0:00:00
  Attempting uninstall: torch
    Found existing installation: torch 2.10.0+cu128
    Uninstalling torch-2.10.0+cu128:
      Successfully uninstalled torch-2.10.0+cu128

Device: cuda
GPU: Tesla P100-PCIE-16GB
Compute capability: (6, 0)

Running CUDA smoke test...
CUDA SMOKE TEST: PASSED


In [3]:
def find_competition_files():

    root = Path(
        "/kaggle/input/competitions/"
        "nppe-2-t-2-26-ecg-heartbeat-arrhythmia-classification/"
        "nppe2_dataset"
    )
    train_path = (
        root / "train.csv"
    )
    test_path = (
        root / "test.csv"
    )
    sample_path = (
        root / "sample_submission.csv"
    )
    if (
        train_path.exists()
        and test_path.exists()
    ):

        return (
            train_path,
            test_path,
            sample_path
        )
    for p in Path(
        "/kaggle/input"
    ).rglob("train.csv"):

        try:

            cols = pd.read_csv(
                p,
                nrows=2
            ).columns.tolist()

            if (
                "label" in cols
                and "id" in cols
                and "sig_249" in cols
            ):

                root = p.parent

                return (
                    p,
                    root / "test.csv",
                    root / "sample_submission.csv"
                )

        except Exception:
            pass


    raise FileNotFoundError(
        "Could not locate competition data."
    )
TRAIN_PATH, TEST_PATH, SAMPLE_SUB_PATH = (
    find_competition_files()
)
print(
    "TRAIN :",
    TRAIN_PATH
)

print(
    "TEST  :",
    TEST_PATH
)

print(
    "SAMPLE:",
    SAMPLE_SUB_PATH
)

train_df = pd.read_csv(
    TRAIN_PATH
)

test_df = pd.read_csv(
    TEST_PATH
)
signal_cols = [
    f"sig_{i}"
    for i in range(250)
]

timing_cols = [
    "pre_rr",
    "post_rr",
    "rr_ratio"
]


required_train = (
    ["id", "label"]
    + signal_cols
    + timing_cols
)


required_test = (
    ["id"]
    + signal_cols
    + timing_cols
)


missing_train = [
    c
    for c in required_train
    if c not in train_df.columns
]


missing_test = [
    c
    for c in required_test
    if c not in test_df.columns
]


if missing_train:

    raise ValueError(
        f"Missing train columns: {missing_train}"
    )


if missing_test:

    raise ValueError(
        f"Missing test columns: {missing_test}"
    )


print(
    "\nTrain shape:",
    train_df.shape
)

print(
    "Test shape:",
    test_df.shape
)
print(
    "\nLabels:"
)

print(
    train_df[
        "label"
    ]
    .value_counts()
    .sort_index()
)


print(
    "\nTiming missingness:"
)

print(
    pd.concat(
        [

            train_df[
                timing_cols
            ]
            .isna()
            .sum()
            .rename(
                "train_missing"
            ),

            test_df[
                timing_cols
            ]
            .isna()
            .sum()
            .rename(
                "test_missing"
            )

        ],
        axis=1
    )
)
if SAMPLE_SUB_PATH.exists():

    sample_sub = pd.read_csv(
        SAMPLE_SUB_PATH
    )

else:

    sample_sub = None


print(
    "\nSample submission:"
)

if sample_sub is not None:

    print(
        sample_sub.head()
    )

    print(
        sample_sub.columns.tolist()
    )

TRAIN : /kaggle/input/competitions/nppe-2-t-2-26-ecg-heartbeat-arrhythmia-classification/nppe2_dataset/train.csv
TEST  : /kaggle/input/competitions/nppe-2-t-2-26-ecg-heartbeat-arrhythmia-classification/nppe2_dataset/test.csv
SAMPLE: /kaggle/input/competitions/nppe-2-t-2-26-ecg-heartbeat-arrhythmia-classification/nppe2_dataset/sample_submission.csv

Train shape: (71748, 255)
Test shape: (68914, 254)

Labels:
label
0    45000
1     3599
2    22552
3      597
Name: count, dtype: int64

Timing missingness:
          train_missing  test_missing
pre_rr                7            13
post_rr              17            10
rr_ratio              7            13

Sample submission:
           id  label
0  te_0000000      0
1  te_0000001      0
2  te_0000002      0
3  te_0000003      0
4  te_0000004      0
['id', 'label']


In [4]:
original_labels = sorted(
    train_df[
        "label"
    ]
    .dropna()
    .unique()
    .tolist()
)


label_to_idx = {

    label: idx

    for idx, label
    in enumerate(
        original_labels
    )
}


idx_to_label = {

    idx: label

    for label, idx
    in label_to_idx.items()
}


N_CLASSES = len(
    original_labels
)


train_df[
    "_target_idx"
] = (

    train_df[
        "label"
    ]
    .map(
        label_to_idx
    )
    .astype(
        np.int64
    )
)
if DEBUG_RUN:

    pieces = []


    for label in original_labels:

        part = train_df[
            train_df[
                "label"
            ] == label
        ]


        part = part.sample(

            n=min(
                DEBUG_ROWS_PER_CLASS,
                len(part)
            ),

            random_state=SEED
        )


        pieces.append(
            part
        )


    work_train = (

        pd.concat(
            pieces
        )

        .sample(
            frac=1.0,
            random_state=SEED
        )

        .reset_index(
            drop=True
        )
    )

else:

    work_train = (
        train_df
        .reset_index(
            drop=True
        )
        .copy()
    )


print(
    "Labels:",
    original_labels
)

print(
    "N_CLASSES:",
    N_CLASSES
)

print(
    "Working shape:",
    work_train.shape
)

print(
    "\nWorking class distribution:"
)

print(
    work_train[
        "_target_idx"
    ]
    .value_counts()
    .sort_index()
)

Labels: [0, 1, 2, 3]
N_CLASSES: 4
Working shape: (71748, 256)

Working class distribution:
_target_idx
0    45000
1     3599
2    22552
3      597
Name: count, dtype: int64


In [5]:
class ECGDataset(
    Dataset
):

    def __init__(
        self,
        df,
        timing_median,
        timing_mean,
        timing_std,
        train_mode=False
    ):

        super().__init__()


        self.df = (
            df
            .reset_index(
                drop=True
            )
            .copy()
        )


        self.train_mode = (
            train_mode
        )
        signals = self.df[
            signal_cols
        ].to_numpy(
            dtype=np.float32,
            copy=True
        )
        signals = np.nan_to_num(
            signals,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )
        self.signals = (
            torch.from_numpy(
                signals
            )
        )
        timing_raw = self.df[
            timing_cols
        ].to_numpy(
            dtype=np.float32,
            copy=True
        )


        missing = (
            np.isnan(
                timing_raw
            )
            .astype(
                np.float32
            )
        )
        timing_filled = np.where(

            np.isnan(
                timing_raw
            ),

            timing_median[
                None,
                :
            ],

            timing_raw
        )
        timing_z = (

            timing_filled
            - timing_mean[
                None,
                :
            ]

        ) / np.maximum(

            timing_std[
                None,
                :
            ],

            1e-6
        )
        pre = (
            timing_filled[:, 0]
        )
        post = (
            timing_filled[:, 1]
        )

        ratio = (
            timing_filled[:, 2]
        )


        engineered = np.stack(
            [

                np.log1p(
                    np.clip(
                        pre,
                        0,
                        None
                    )
                ),

                np.log1p(
                    np.clip(
                        post,
                        0,
                        None
                    )
                ),

                np.log(
                    np.clip(
                        ratio,
                        1e-3,
                        None
                    )
                ),

                post - pre,

                missing[:, 0],
                missing[:, 1],
                missing[:, 2]

            ],
            axis=1
        ).astype(
            np.float32
        )


        timing_features = np.concatenate(
            [

                timing_z,

                engineered

            ],
            axis=1
        )


        self.timing = (
            torch.from_numpy(
                timing_features
            )
        )


        self.targets = (
            torch.from_numpy(
                self.df[
                    "_target_idx"
                ]
                .to_numpy(
                    np.int64
                )
            )
        )


    def __len__(
        self
    ):

        return len(
            self.signals
        )


    @staticmethod
    def shift_signal(
        x,
        shift
    ):

        if shift == 0:

            return x


        y = torch.zeros_like(
            x
        )


        if shift > 0:

            y[
                shift:
            ] = x[
                :-shift
            ]

        else:

            y[
                :shift
            ] = x[
                -shift:
            ]


        return y


    def augment(
        self,
        x
    ):

        # amplitude jitter
        if torch.rand(()) < 0.55:

            gain = (
                1.0
                + 0.025
                * torch.randn(())
            )

            x = (
                x * gain
            )


        # small noise
        if torch.rand(()) < 0.30:

            x = (
                x
                + 0.01
                * torch.randn_like(
                    x
                )
            )


        # small time shift
        if torch.rand(()) < 0.18:

            shift = int(
                torch.randint(
                    -2,
                    3,
                    (1,)
                ).item()
            )

            x = self.shift_signal(
                x,
                shift
            )


        # small local mask
        if torch.rand(()) < 0.15:

            width = int(
                torch.randint(
                    2,
                    7,
                    (1,)
                ).item()
            )


            start = int(
                torch.randint(
                    0,
                    250 - width + 1,
                    (1,)
                ).item()
            )


            x[
                start:
                start + width
            ] = 0.0


        return x


    def __getitem__(
        self,
        index
    ):

        x = (
            self.signals[
                index
            ].clone()
        )


        if (
            self.train_mode
            and AUGMENT
        ):

            x = self.augment(
                x
            )


        return (

            {
                "signal":
                    x.unsqueeze(0),

                "timing":
                    self.timing[
                        index
                    ]
            },

            self.targets[
                index
            ]
        )


class ECGTestDataset(
    Dataset
):

    def __init__(
        self,
        df,
        timing_median,
        timing_mean,
        timing_std
    ):

        super().__init__()


        signals = self.df_to_signals(
            df
        )


        self.signals = signals


        timing_raw = df[
            timing_cols
        ].to_numpy(
            dtype=np.float32,
            copy=True
        )


        missing = (
            np.isnan(
                timing_raw
            )
            .astype(
                np.float32
            )
        )


        timing_filled = np.where(

            np.isnan(
                timing_raw
            ),

            timing_median[
                None,
                :
            ],

            timing_raw
        )


        timing_z = (

            timing_filled
            - timing_mean[
                None,
                :
            ]

        ) / np.maximum(

            timing_std[
                None,
                :
            ],

            1e-6
        )


        pre = timing_filled[:, 0]
        post = timing_filled[:, 1]
        ratio = timing_filled[:, 2]


        engineered = np.stack(
            [

                np.log1p(
                    np.clip(
                        pre,
                        0,
                        None
                    )
                ),

                np.log1p(
                    np.clip(
                        post,
                        0,
                        None
                    )
                ),

                np.log(
                    np.clip(
                        ratio,
                        1e-3,
                        None
                    )
                ),

                post - pre,

                missing[:, 0],
                missing[:, 1],
                missing[:, 2]

            ],
            axis=1
        ).astype(
            np.float32
        )


        self.timing = torch.from_numpy(
            np.concatenate(
                [
                    timing_z,
                    engineered
                ],
                axis=1
            )
        )


    @staticmethod
    def df_to_signals(
        df
    ):

        x = df[
            signal_cols
        ].to_numpy(
            dtype=np.float32,
            copy=True
        )


        x = np.nan_to_num(
            x,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )


        return torch.from_numpy(
            x
        )


    def __len__(
        self
    ):

        return len(
            self.signals
        )


    def __getitem__(
        self,
        index
    ):

        return {

            "signal":
                self.signals[
                    index
                ].unsqueeze(0),

            "timing":
                self.timing[
                    index
                ]

        }
def make_loader(
    dataset,
    shuffle=False
):

    return DataLoader(

        dataset,

        batch_size=BATCH_SIZE,

        shuffle=shuffle,

        num_workers=NUM_WORKERS,

        pin_memory=True,

        persistent_workers=(
            NUM_WORKERS > 0
        ),

        drop_last=False
    )

In [6]:
class SEBlock(
    nn.Module
):
    def __init__(
        self,
        channels,
        reduction=8
    ):

        super().__init__()


        hidden = max(
            channels // reduction,
            8
        )
        self.pool = (
            nn.AdaptiveAvgPool1d(1)
        )


        self.fc = nn.Sequential(

            nn.Conv1d(
                channels,
                hidden,
                1
            ),

            nn.GELU(),

            nn.Conv1d(
                hidden,
                channels,
                1
            ),

            nn.Sigmoid()
        )


    def forward(
        self,
        x
    ):

        return (
            x
            * self.fc(
                self.pool(x)
            )
        )
class MultiScaleResBlock(
    nn.Module
):

    def __init__(
        self,
        channels,
        dilation=1,
        dropout=0.08
    ):

        super().__init__()
        self.branch3 = nn.Sequential(
            nn.Conv1d(
                channels,
                channels,
                3,
                padding=dilation,
                dilation=dilation,
                bias=False
            ),
            nn.BatchNorm1d(
                channels
            ),
            nn.GELU()
        )
        self.branch5 = nn.Sequential(

            nn.Conv1d(
                channels,
                channels,
                5,
                padding=2 * dilation,
                dilation=dilation,
                bias=False
            ),
            nn.BatchNorm1d(
                channels
            ),
            nn.GELU()
        )
        self.branch7 = nn.Sequential(
            nn.Conv1d(
                channels,
                channels,
                7,
                padding=3 * dilation,
                dilation=dilation,
                bias=False
            ),
            nn.BatchNorm1d(
                channels
            ),
            nn.GELU()
        )
        self.mix = nn.Sequential(

            nn.Conv1d(
                channels * 3,
                channels,
                1,
                bias=False
            ),
            nn.BatchNorm1d(
                channels
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            )
        )
        self.se = SEBlock(
            channels
        )
    def forward(
        self,
        x
    ):
        y = torch.cat(
            [
                self.branch3(x),
                self.branch5(x),
                self.branch7(x)
            ],
            dim=1
        )
        y = self.mix(
            y
        )
        y = self.se(
            y
        )
        return (
            x + y
        )
class DownBlock(
    nn.Module
):

    def __init__(
        self,
        in_ch,
        out_ch
    ):

        super().__init__()


        self.net = nn.Sequential(

            nn.Conv1d(
                in_ch,
                out_ch,
                5,
                stride=2,
                padding=2,
                bias=False
            ),
            nn.BatchNorm1d(
                out_ch
            ),
            nn.GELU()
        )
    def forward(
        self,
        x
    ):
        return self.net(
            x
        )
class ECGRhythmNet(
    nn.Module
):

    def __init__(
        self,
        n_classes
    ):

        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(
                2,
                64,
                7,
                padding=3,
                bias=False
            ),
            nn.BatchNorm1d(
                64
            ),
            nn.GELU()
        )
        self.stage1 = nn.Sequential(

            MultiScaleResBlock(
                64,
                1
            ),

            MultiScaleResBlock(
                64,
                1
            ),

            MultiScaleResBlock(
                64,
                2
            )
        )


        self.down1 = DownBlock(
            64,
            96
        )


        self.stage2 = nn.Sequential(

            MultiScaleResBlock(
                96,
                1
            ),

            MultiScaleResBlock(
                96,
                2
            ),

            MultiScaleResBlock(
                96,
                3
            )
        )


        self.down2 = DownBlock(
            96,
            160
        )


        self.stage3 = nn.Sequential(

            MultiScaleResBlock(
                160,
                1
            ),

            MultiScaleResBlock(
                160,
                2
            ),

            MultiScaleResBlock(
                160,
                4
            )
        )
        transformer_layer = (
            nn.TransformerEncoderLayer(
                d_model=160,
                nhead=8,
                dim_feedforward=320,
                dropout=0.10,
                activation="gelu",
                batch_first=True,
                norm_first=True
            )
        )


        self.transformer = (
            nn.TransformerEncoder(
                transformer_layer,
                num_layers=3
            )
        )


        self.positional_embedding = nn.Parameter(
            torch.zeros(
                1,
                64,
                160
            )
        )
        self.token_score = nn.Sequential(

            nn.LayerNorm(
                160
            ),
            nn.Linear(
                160,
                1
            )
        )
        self.timing_mlp = nn.Sequential(

            nn.Linear(
                10,
                64
            ),

            nn.LayerNorm(
                64
            ),

            nn.GELU(),

            nn.Dropout(
                0.10
            ),

            nn.Linear(
                64,
                64
            ),

            nn.GELU()
        )


        # ----------------------------------------------------
        # Raw signal statistics
        # ----------------------------------------------------

        self.stats_mlp = nn.Sequential(

            nn.Linear(
                8,
                32
            ),

            nn.GELU()
        )

        self.timing_to_signal = nn.Linear(
            64,
            160
        )
        self.gate = nn.Sequential(

            nn.Linear(
                224,
                160
            ),
            nn.GELU(),
            nn.Linear(
                160,
                160
            ),

            nn.Sigmoid()
        )

        self.head = nn.Sequential(

            nn.LayerNorm(
                256
            ),

            nn.Linear(
                256,
                128
            ),

            nn.GELU(),

            nn.Dropout(
                0.20
            ),

            nn.Linear(
                128,
                n_classes
            )
        )


        self.initialize()


    def initialize(
        self
    ):

        for module in self.modules():

            if isinstance(
                module,
                nn.Conv1d
            ):

                nn.init.kaiming_normal_(
                    module.weight,
                    mode="fan_out",
                    nonlinearity="relu"
                )


            elif isinstance(
                module,
                nn.Linear
            ):

                nn.init.trunc_normal_(
                    module.weight,
                    std=0.02
                )


                if module.bias is not None:

                    nn.init.zeros_(
                        module.bias
                    )


            elif isinstance(
                module,
                nn.BatchNorm1d
            ):

                nn.init.ones_(
                    module.weight
                )

                nn.init.zeros_(
                    module.bias
                )


    @staticmethod
    def signal_statistics(
        x
    ):

        mean = x.mean(
            dim=-1
        )

        std = x.std(
            dim=-1
        )

        maximum = x.amax(
            dim=-1
        )

        minimum = x.amin(
            dim=-1
        )

        energy = (
            x.square()
            .mean(
                dim=-1
            )
            .sqrt()
        )

        diff = (
            x[..., 1:]
            - x[..., :-1]
        )


        mean_abs_diff = (
            diff.abs()
            .mean(
                dim=-1
            )
        )


        max_abs = (
            x.abs()
            .amax(
                dim=-1
            )
        )


        zero_crossings = (

            (
                x[..., 1:]
                * x[..., :-1]
                < 0
            )

            .float()
            .mean(
                dim=-1
            )
        )


        return torch.cat(
            [

                mean,
                std,
                maximum,
                minimum,
                energy,
                mean_abs_diff,
                max_abs,
                zero_crossings

            ],
            dim=1
        )


    def forward(
        self,
        batch
    ):

        signal = batch[
            "signal"
        ]

        derivative = (
            signal[..., 1:]
            - signal[..., :-1]
        )


        derivative = F.pad(
            derivative,
            (1, 0)
        )


        x = torch.cat(
            [
                signal,
                derivative
            ],
            dim=1
        )


        raw_signal = signal

        x = self.stem(
            x
        )
        x = self.stage1(
            x
        )
        x = self.down1(
            x
        )
        x = self.stage2(
            x
        )
        x = self.down2(
            x
        )
        x = self.stage3(
            x
        )

        x = x.transpose(
            1,
            2
        )
        x = (

            x

            + self.positional_embedding[
                :,
                :x.shape[1]
            ]
        )

        x = self.transformer(
            x
        )

        scores = (
            self.token_score(
                x
            )
            .squeeze(-1)
        )


        attention = torch.softmax(
            scores,
            dim=1
        )


        signal_embedding = (
            x
            * attention.unsqueeze(-1)
        ).sum(
            dim=1
        )

        timing_embedding = (
            self.timing_mlp(
                batch[
                    "timing"
                ]
            )
        )

        gate_input = torch.cat(
            [

                signal_embedding,

                timing_embedding

            ],
            dim=1
        )


        gate = self.gate(
            gate_input
        )


        signal_embedding = (

            signal_embedding

            + gate

            * self.timing_to_signal(
                timing_embedding
            )
        )

        stats_embedding = (
            self.stats_mlp(

                self.signal_statistics(
                    raw_signal
                )

            )
        )

        features = torch.cat(
            [

                signal_embedding,

                timing_embedding,

                stats_embedding

            ],
            dim=1
        )


        return self.head(
            features
        )

In [7]:
def unpack_batch(batch):
    if isinstance(batch, (tuple, list)):
        features = batch[0]
        labels = batch[1] if len(batch) > 1 else None

    elif isinstance(batch, dict):
        features = batch
        labels = None

    else:
        raise TypeError(
            f"Unexpected batch type: {type(batch)}"
        )

    return features, labels


def move_features_to_device(features):
    return {
        key: value.to(
            DEVICE,
            non_blocking=True
        )
        for key, value in features.items()
    }
def effective_num_weights(
    y,
    beta=0.9995
):

    counts = np.bincount(
        y,
        minlength=N_CLASSES
    ).astype(
        np.float64
    )

    effective = (
        1.0
        - np.power(
            beta,
            counts
        )
    )

    weights = (
        (1.0 - beta)
        / np.maximum(
            effective,
            1e-12
        )
    )

    weights = (
        weights
        / weights.mean()
    )

    return (
        torch.tensor(
            weights,
            dtype=torch.float32,
            device=DEVICE
        ),
        counts
    )


class ClassBalancedCE(nn.Module):

    def __init__(
        self,
        weights,
        smoothing=0.04
    ):

        super().__init__()

        self.register_buffer(
            "weights",
            weights
        )

        self.smoothing = smoothing


    def forward(
        self,
        logits,
        target
    ):

        return F.cross_entropy(
            logits,
            target,
            weight=self.weights,
            label_smoothing=self.smoothing
        )

class ModelEMA:

    def __init__(
        self,
        model,
        decay=0.995
    ):

        self.decay = decay

        self.shadow = {}

        for key, value in model.state_dict().items():

            if value.dtype.is_floating_point:

                self.shadow[key] = (
                    value.detach().clone()
                )
        self.backup = {}


    @torch.no_grad()
    def update(
        self,
        model
    ):

        state = model.state_dict()

        for key in self.shadow:

            self.shadow[key].mul_(
                self.decay
            )

            self.shadow[key].add_(
                state[key].detach(),
                alpha=(
                    1.0
                    - self.decay
                )
            )


    @torch.no_grad()
    def apply_to(
        self,
        model
    ):

        self.backup = {}

        state = model.state_dict()

        for key in self.shadow:

            self.backup[key] = (
                state[key]
                .detach()
                .clone()
            )

            state[key].copy_(
                self.shadow[key]
            )


    @torch.no_grad()
    def restore(
        self,
        model
    ):

        state = model.state_dict()

        for key, value in self.backup.items():

            state[key].copy_(
                value
            )

        self.backup = {}


    def state_dict(self):

        return {
            key: value.detach().cpu()
            for key, value in self.shadow.items()
        }


    def load_state_dict(
        self,
        state_dict,
        model
    ):

        model_state = model.state_dict()

        with torch.no_grad():

            for key, value in state_dict.items():

                if key in model_state:

                    model_state[key].copy_(
                        value.to(
                            device=model_state[key].device,
                            dtype=model_state[key].dtype
                        )
                    )

@torch.no_grad()
def predict_logits(
    model,
    loader
):

    model.eval()

    outputs = []

    for raw_batch in loader:

        features, _ = unpack_batch(
            raw_batch
        )

        features = move_features_to_device(
            features
        )

        with torch.amp.autocast(
            "cuda",
            dtype=torch.float16,
            enabled=(
                AMP
                and DEVICE.type == "cuda"
            )
        ):

            logits = model(
                features
            )

        outputs.append(
            logits
            .float()
            .cpu()
        )

    return torch.cat(
        outputs,
        dim=0
    ).numpy()

def macro_f1(
    logits,
    labels
):

    predictions = (
        logits
        .argmax(
            axis=1
        )
    )

    return f1_score(
        labels,
        predictions,
        average="macro"
    )

def make_scheduler(
    optimizer,
    total_steps,
    warmup_steps
):

    def schedule(step):

        if step < warmup_steps:

            return max(
                1e-3,
                (step + 1)
                / max(
                    1,
                    warmup_steps
                )
            )


        progress = (
            step
            - warmup_steps
        ) / max(
            1,
            total_steps
            - warmup_steps
        )


        return (
            0.05
            + 0.95
            * 0.5
            * (
                1.0
                + math.cos(
                    math.pi
                    * progress
                )
            )
        )


    return torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        schedule
    )

In [8]:
def train_one_fold(
    model,
    train_loader,
    val_loader,
    criterion,
    fold,
    epochs,
    patience
):

    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=LR,

        weight_decay=WEIGHT_DECAY,

        betas=(
            0.9,
            0.99
        )
    )


    total_steps = (
        epochs
        * len(train_loader)
    )


    scheduler = make_scheduler(

        optimizer,

        total_steps,

        warmup_steps=max(
            100,
            len(train_loader)
        )
    )


    scaler = torch.amp.GradScaler(

        "cuda",

        enabled=(
            AMP
            and DEVICE.type == "cuda"
        )
    )


    ema = ModelEMA(
        model,
        decay=0.995
    )


    best_f1 = -1.0
    best_epoch = -1

    best_ema_state = None

    patience_count = 0


    for epoch in range(
        1,
        epochs + 1
    ):

        model.train()

        total_loss = 0.0
        total_items = 0

        t0 = time.time()


        for raw_batch in train_loader:

            features, labels = unpack_batch(
                raw_batch
            )


            features = move_features_to_device(
                features
            )


            labels = labels.to(
                DEVICE,
                non_blocking=True
            )


            optimizer.zero_grad(
                set_to_none=True
            )


            with torch.amp.autocast(

                "cuda",

                dtype=torch.float16,

                enabled=(
                    AMP
                    and DEVICE.type == "cuda"
                )

            ):

                logits = model(
                    features
                )


                loss = criterion(
                    logits,
                    labels
                )


            scaler.scale(
                loss
            ).backward()


            scaler.unscale_(
                optimizer
            )


            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                GRAD_CLIP
            )


            scaler.step(
                optimizer
            )


            scaler.update()


            scheduler.step()


            ema.update(
                model
            )


            batch_size = (
                labels.shape[0]
            )


            total_loss += (
                loss.item()
                * batch_size
            )


            total_items += (
                batch_size
            )

        ema.apply_to(
            model
        )


        val_logits = predict_logits(
            model,
            val_loader
        )


        val_labels_list = []


        for raw_batch in val_loader:

            _, labels = unpack_batch(
                raw_batch
            )

            val_labels_list.append(
                labels
            )


        val_labels = (
            torch.cat(
                val_labels_list,
                dim=0
            )
            .numpy()
        )


        val_f1 = macro_f1(
            val_logits,
            val_labels
        )


        val_accuracy = (

            val_logits.argmax(
                axis=1
            )

            == val_labels

        ).mean()

        if (
            val_f1
            > best_f1
            + 1e-6
        ):

            best_f1 = val_f1

            best_epoch = epoch

            best_ema_state = (
                ema.state_dict()
            )

            patience_count = 0

        else:

            patience_count += 1


        ema.restore(
            model
        )


        elapsed = (
            time.time()
            - t0
        )


        average_loss = (
            total_loss
            / max(
                total_items,
                1
            )
        )


        print(

            f"Fold {fold} | "
            f"Epoch {epoch:02d}/{epochs} | "
            f"loss={average_loss:.5f} | "
            f"val_F1={val_f1:.5f} | "
            f"val_acc={val_accuracy:.5f} | "
            f"lr={optimizer.param_groups[0]['lr']:.2e} | "
            f"{elapsed:.1f}s"

        )


        if patience_count >= patience:
            print(
                "Early stopping."
            )
            break

    ema.load_state_dict(
        best_ema_state,
        model
    )


    print(

        f"\nFold {fold} best F1 = "
        f"{best_f1:.5f} "
        f"at epoch {best_epoch}\n"

    )


    return (

        model,

        float(
            best_f1
        ),

        int(
            best_epoch
        )

    )

if DEBUG_RUN:

    run_epochs = DEBUG_EPOCHS

    run_folds = 2

else:

    run_epochs = EPOCHS

    run_folds = N_FOLDS


y_all = (
    work_train[
        "_target_idx"
    ]
    .to_numpy(
        np.int64
    )
)


oof_logits = np.zeros(

    (
        len(work_train),
        N_CLASSES
    ),

    dtype=np.float32

)


oof_filled = np.zeros(

    len(work_train),

    dtype=bool

)


fold_models = []

fold_scores = []


skf = StratifiedKFold(

    n_splits=N_FOLDS,

    shuffle=True,

    random_state=SEED
)


for fold, (
    train_idx,
    val_idx
) in enumerate(

    skf.split(

        np.zeros(
            len(work_train)
        ),

        y_all

    ),

    start=1
):

    if fold > run_folds:

        break


    print(
        "\n"
        + "=" * 80
    )

    print(
        f"FOLD {fold}/{run_folds}"
    )

    print(
        "=" * 80
    )


    fold_train = (
        work_train
        .iloc[
            train_idx
        ]
        .reset_index(
            drop=True
        )
    )


    fold_val = (
        work_train
        .iloc[
            val_idx
        ]
        .reset_index(
            drop=True
        )
    )

    timing_median = (
        fold_train[
            timing_cols
        ]
        .median()
        .to_numpy(
            np.float32
        )
    )


    timing_filled = (
        fold_train[
            timing_cols
        ]
        .fillna(
            pd.Series(
                timing_median,
                index=timing_cols
            )
        )
    )


    timing_mean = (
        timing_filled
        .mean()
        .to_numpy(
            np.float32
        )
    )


    timing_std = (
        timing_filled
        .std()
        .to_numpy(
            np.float32
        )
    )


    timing_std = np.where(

        np.isfinite(
            timing_std
        )
        &
        (
            timing_std
            > 1e-6
        ),

        timing_std,

        1.0

    ).astype(
        np.float32
    )

    train_dataset = ECGDataset(

        fold_train,

        timing_median,

        timing_mean,

        timing_std,

        train_mode=True
    )


    val_dataset = ECGDataset(

        fold_val,

        timing_median,

        timing_mean,

        timing_std,

        train_mode=False
    )


    train_loader = make_loader(
        train_dataset,
        shuffle=True
    )


    val_loader = make_loader(
        val_dataset,
        shuffle=False
    )

    class_weights, counts = (
        effective_num_weights(

            fold_train[
                "_target_idx"
            ]
            .to_numpy(
                np.int64
            )
        )
    )


    print(
        "Class counts:",
        counts.astype(
            int
        ).tolist()
    )


    print(
        "Class weights:",
        class_weights
        .detach()
        .cpu()
        .numpy()
        .round(3)
        .tolist()
    )

    model = (
        ECGRhythmNet(
            N_CLASSES
        )
        .to(
            DEVICE
        )
    )


    criterion = (
        ClassBalancedCE(
            class_weights,
            LABEL_SMOOTHING
        )
        .to(
            DEVICE
        )
    )

    model, score, best_epoch = (
        train_one_fold(

            model,

            train_loader,

            val_loader,

            criterion,

            fold,

            run_epochs,

            PATIENCE
        )
    )
    fold_val_logits = predict_logits(
        model,
        val_loader
    )


    oof_logits[
        val_idx
    ] = fold_val_logits


    oof_filled[
        val_idx
    ] = True

    fold_models.append(

        {

            "model":
                model,

            "timing_median":
                timing_median,

            "timing_mean":
                timing_mean,

            "timing_std":
                timing_std,

            "fold":
                fold,

            "score":
                score

        }
    )


    fold_scores.append(
        score
    )
    del (
        train_loader,
        val_loader,
        train_dataset,
        val_dataset,
        criterion,
        fold_train,
        fold_val
    )


    gc.collect()

    torch.cuda.empty_cache()
mask = oof_filled


oof_y = (
    y_all[
        mask
    ]
)


oof_logits_used = (
    oof_logits[
        mask
    ]
)


oof_predictions = (
    oof_logits_used
    .argmax(
        axis=1
    )
)


oof_f1 = f1_score(

    oof_y,

    oof_predictions,

    average="macro"
)


print(
    "\n"
    + "=" * 80
)

print(
    "OOF RESULTS"
)

print(
    "=" * 80
)


print(
    "Fold F1:",
    [
        round(
            float(x),
            5
        )
        for x in fold_scores
    ]
)


print(
    "OOF macro F1:",
    round(
        oof_f1,
        6
    )
)


print(
    "\nClassification report:"
)


print(
    classification_report(
        oof_y,
        oof_predictions,
        digits=4
    )
)


print(
    "\nConfusion matrix:"
)


print(
    confusion_matrix(
        oof_y,
        oof_predictions
    )
)


FOLD 1/3
Class counts: [30000, 2400, 15034, 398]
Class weights: [0.44600000977516174, 0.6380000114440918, 0.44600000977516174, 2.4700000286102295]
Fold 1 | Epoch 01/18 | loss=0.58793 | val_F1=0.42527 | val_acc=0.84308 | lr=2.00e-03 | 71.0s
Fold 1 | Epoch 02/18 | loss=0.42547 | val_F1=0.63548 | val_acc=0.94861 | lr=1.98e-03 | 35.8s
Fold 1 | Epoch 03/18 | loss=0.40955 | val_F1=0.85733 | val_acc=0.97897 | lr=1.94e-03 | 35.8s
Fold 1 | Epoch 04/18 | loss=0.39838 | val_F1=0.90803 | val_acc=0.98725 | lr=1.86e-03 | 35.8s
Fold 1 | Epoch 05/18 | loss=0.38643 | val_F1=0.91614 | val_acc=0.98808 | lr=1.75e-03 | 35.7s
Fold 1 | Epoch 06/18 | loss=0.38181 | val_F1=0.92452 | val_acc=0.98980 | lr=1.62e-03 | 35.8s
Fold 1 | Epoch 07/18 | loss=0.37382 | val_F1=0.92765 | val_acc=0.99030 | lr=1.47e-03 | 35.8s
Fold 1 | Epoch 08/18 | loss=0.36875 | val_F1=0.92372 | val_acc=0.99026 | lr=1.31e-03 | 36.0s
Fold 1 | Epoch 09/18 | loss=0.36337 | val_F1=0.92929 | val_acc=0.99093 | lr=1.14e-03 | 36.3s
Fold 1 | Epoch 

In [9]:
def tune_class_bias(
    logits,
    labels
):

    bias = np.zeros(
        logits.shape[1],
        dtype=np.float32
    )


    best_score = f1_score(

        labels,

        (
            logits + bias
        ).argmax(
            axis=1
        ),

        average="macro"
    )


    search_grids = [

        np.linspace(
            -1.00,
            1.00,
            41
        ),

        np.linspace(
            -0.25,
            0.25,
            41
        ),

        np.linspace(
            -0.08,
            0.08,
            33
        )

    ]


    for grid in search_grids:

        changed = True


        while changed:

            changed = False


            for class_id in range(
                logits.shape[1]
            ):

                local_best = best_score

                local_value = (
                    bias[class_id]
                )


                for delta in grid:

                    candidate = (
                        bias.copy()
                    )


                    candidate[
                        class_id
                    ] = (

                        bias[
                            class_id
                        ]

                        + np.float32(
                            delta
                        )

                    )


                    prediction = (

                        logits
                        + candidate

                    ).argmax(
                        axis=1
                    )


                    score = f1_score(

                        labels,

                        prediction,

                        average="macro"
                    )


                    if (
                        score
                        > local_best
                        + 1e-7
                    ):

                        local_best = score

                        local_value = (
                            candidate[
                                class_id
                            ]
                        )


                if (
                    local_best
                    > best_score
                    + 1e-7
                ):

                    bias[
                        class_id
                    ] = local_value

                    best_score = (
                        local_best
                    )

                    changed = True


    return (
        bias,
        best_score
    )


oof_bias, calibrated_f1 = (
    tune_class_bias(
        oof_logits_used,
        oof_y
    )
)


print(
    "Raw OOF F1:",
    oof_f1
)


print(
    "Calibrated OOF F1:",
    calibrated_f1
)


print(
    "Class bias:",
    oof_bias
)

Raw OOF F1: 0.9297658822479231
Calibrated OOF F1: 0.9328004225033321
Class bias: [0.3375 0.05   0.4625 0.    ]


In [10]:
def infer_test_logits(
    model_info
):

    model = (
        model_info[
            "model"
        ]
    )


    timing_median = (
        model_info[
            "timing_median"
        ]
    )


    timing_mean = (
        model_info[
            "timing_mean"
        ]
    )


    timing_std = (
        model_info[
            "timing_std"
        ]
    )


    test_dataset = ECGTestDataset(

        test_df,

        timing_median,

        timing_mean,

        timing_std
    )


    test_loader = make_loader(

        test_dataset,

        shuffle=False
    )


    model.eval()


    @torch.no_grad()
    def predict_with_gain(
        gain
    ):

        outputs = []


        for raw_batch in test_loader:

            features, _ = unpack_batch(
                raw_batch
            )


            features = move_features_to_device(
                features
            )


            if gain != 1.0:

                features[
                    "signal"
                ] = (

                    features[
                        "signal"
                    ]

                    * gain

                )


            with torch.amp.autocast(

                "cuda",

                dtype=torch.float16,

                enabled=(
                    AMP
                    and DEVICE.type == "cuda"
                )

            ):

                logits = model(
                    features
                )


            outputs.append(
                logits
                .float()
                .cpu()
            )


        return torch.cat(
            outputs,
            dim=0
        ).numpy()


    if TTA:

        gains = [
            1.00,
            0.99,
            1.01
        ]

    else:

        gains = [
            1.00
        ]


    all_logits = []


    for gain in gains:

        print(
            f"TTA gain = {gain}"
        )

        all_logits.append(
            predict_with_gain(
                gain
            )
        )


    return np.mean(
        all_logits,
        axis=0
    )

test_fold_logits = []


for model_info in fold_models:

    print(
        "\n"
        + "=" * 70
    )

    print(
        "TEST INFERENCE — FOLD",
        model_info["fold"]
    )

    print(
        "=" * 70
    )


    logits = infer_test_logits(
        model_info
    )


    test_fold_logits.append(
        logits
    )

test_logits = np.mean(
    test_fold_logits,
    axis=0
)

test_calibrated_logits = (

    test_logits

    + oof_bias[
        None,
        :
    ]

)


test_prediction_indices = (
    test_calibrated_logits
    .argmax(
        axis=1
    )
)


test_predictions = np.array(

    [

        idx_to_label[
            int(idx)
        ]

        for idx in test_prediction_indices

    ]
)

submission = pd.DataFrame(

    {

        "id":
            test_df[
                "id"
            ].astype(
                str
            ),

        "label":
            test_predictions

    }

)

if sample_sub is not None:

    if set(
        sample_sub.columns
    ) == set(
        submission.columns
    ):

        submission = (
            submission[
                sample_sub.columns
            ]
        )

assert (
    len(submission)
    == len(test_df)
)


assert (
    submission[
        "id"
    ].isna().sum()
    == 0
)


assert (
    submission[
        "label"
    ].isna().sum()
    == 0
)


assert (

    submission[
        "id"
    ]
    .astype(str)
    .tolist()

    ==

    test_df[
        "id"
    ]
    .astype(str)
    .tolist()

)


assert set(
    submission[
        "label"
    ].unique()
).issubset(
    set(original_labels)
)

submission.to_csv(

    SUBMISSION_PATH,

    index=False

)

if not SUBMISSION_PATH.exists():

    raise RuntimeError(
        "submission.csv was not created."
    )


print(
    "\n"
    + "=" * 80
)

print(
    "SUBMISSION READY"
)

print(
    "=" * 80
)


print(
    "File:",
    SUBMISSION_PATH
)


print(
    "Rows:",
    len(submission)
)


print(
    "\nPrediction distribution:"
)


print(
    submission[
        "label"
    ]
    .value_counts()
    .sort_index()
)


print(
    "\nFirst rows:"
)


print(
    submission.head()
)


print(
    "\nONLY competition output:"
)

print(
    SUBMISSION_PATH
)


TEST INFERENCE — FOLD 1
TTA gain = 1.0
TTA gain = 0.99
TTA gain = 1.01

TEST INFERENCE — FOLD 2
TTA gain = 1.0
TTA gain = 0.99
TTA gain = 1.01

TEST INFERENCE — FOLD 3
TTA gain = 1.0
TTA gain = 0.99
TTA gain = 1.01

SUBMISSION READY
File: /kaggle/working/submission.csv
Rows: 68914

Prediction distribution:
label
0    60631
1      782
2     7331
3      170
Name: count, dtype: int64

First rows:
           id  label
0  te_0000000      1
1  te_0000001      0
2  te_0000002      0
3  te_0000003      0
4  te_0000004      0

ONLY competition output:
/kaggle/working/submission.csv
